# 01 — Preprocessing
Load raw CSVs → clean → NER + toponym masking → text normalization → sentiment analysis → store in DuckDB.

In [ ]:
import sys
import yaml
from pathlib import Path

PROJECT_ROOT = Path('__file__').resolve().parent.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from reddit_np_topics import db
from reddit_np_topics.preprocessing.cleaner import load_and_clean_all
from reddit_np_topics.preprocessing.ner import NERProcessor
from reddit_np_topics.preprocessing.normalizer import TextNormalizer
from reddit_np_topics.sentiment import SentimentAnalyzer

with open(PROJECT_ROOT / 'config.yml') as f:
    cfg = yaml.safe_load(f)

RAW_FOLDER = PROJECT_ROOT / 'data' / 'raw'
DB_PATH = PROJECT_ROOT / 'data' / 'reddit_np.duckdb'

In [ ]:
# Initialise DB schema (includes sentiment columns)
db.init_schema(DB_PATH)

In [ ]:
# Step 1: Load and clean raw CSVs
df = load_and_clean_all(RAW_FOLDER)
print(df.shape)
df.head()

In [ ]:
# Step 2: NER — detect locations, replace with TOPONYM token
ner_cfg = cfg['preprocessing']
ner = NERProcessor(
    model_name=ner_cfg['ner_model'],
    labels=ner_cfg['ner_labels'],
    threshold=ner_cfg['ner_threshold'],
)
df = ner.process_dataframe(df, text_col='text')
df[['text', 'tokens', 'loc_entities']].head()

In [ ]:
# Step 3: Text normalization
normalizer = TextNormalizer()
df['tokens'] = normalizer.normalize_series(df['tokens'])
df = df.dropna(subset=['tokens'])
df = df[df['tokens'].str.strip() != '']
print(f'{len(df)} rows after normalization')

In [ ]:
# Step 4: Sentiment analysis (on raw text, before normalization strips context)
analyzer = SentimentAnalyzer(batch_size=32)
df = analyzer.analyze_dataframe(df, text_col='text')
df[['text', 'sentiment', 'sentiment_score']].head()

In [ ]:
# Step 5: Write to DuckDB
cols = ['origin_id','post_guid','park_name','user_guid','publish_date',
        'post_thumbnail_url','like_count','post_comment_count','post_url',
        'tags','emoji','post_title','body','text','tokens','loc_entities',
        'sentiment','sentiment_score']
db.write_normalized_posts(df[cols], DB_PATH)
print('Written to normalized_posts')

In [ ]:
# Verify
parks = db.list_parks('normalized_posts', DB_PATH)
print(f'{len(parks)} parks in DB:', parks)